# Ablations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yazanjer/An_Explainable_AI_Education/blob/main/notebooks/04_ablations.ipynb)

**Answers:** Editor comment 5
**Estimated runtime:** 1 h · **Hardware:** CPU
**Quick mode:** set `QUICK_MODE = True` in the setup cell for a fast smoke test.

Five ablations of the proposed method: variable length off, symmetric uncertainty off,
each objective term, initialisation, and length-adaptation direction.

**Note:** `beta_stagnation` must be strictly less than `max_iter` or the
length-adaptation mechanism can never fire and every arm returns identical results.
`VLPSOSelector` raises on that configuration.

---


In [ ]:
# --- Environment setup -------------------------------------------------
# Detects Colab, mounts Drive only when in Colab, installs pinned deps.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
QUICK_MODE = True   # set False for the full budget

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT = Path("/content/drive/MyDrive/An_Explainable_AI_Education")
    PROJECT.mkdir(parents=True, exist_ok=True)
    if not (PROJECT / "src").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/yazanjer/An_Explainable_AI_Education.git", str(PROJECT)],
                       check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT / "requirements.txt")], check=False)
else:
    PROJECT = Path(os.environ.get("VLPSO_PROJECT_ROOT", Path.cwd().parent))

os.environ["VLPSO_PROJECT_ROOT"] = str(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))

from vlpso_xai.config import load_config, set_global_seeds, environment_report
cfg = load_config("quick" if QUICK_MODE else "default")
set_global_seeds(cfg.seed)
cfg.paths.mkdirs()
print("project root:", cfg.paths.root)
print("config:", cfg.config_path.name, "| hash:", cfg.hash()[:12])


In [ ]:
ARMS = {
    "full": dict(),
    "variable_length_off": dict(length_direction="shrink_only"),
    "grow_only": dict(length_direction="grow_only"),
    "su_off": dict(ranking="mutual_info"),
    "no_cardinality_penalty": dict(length_penalty=0.0),
    "no_interpretability_term": dict(interpretability_weight=0.0),
    "random_init": dict(init="random"),
}
rows = []
for arm, override in ARMS.items():
    r = run_nested_cv(X, y, g,
        models=get_models(["LogisticRegression_L2"], fast=True),
        selector_factory=lambda o=override: VLPSOSelector(
            divisions=vp["divisions"], **{**common, **o}),
        cfg=NestedCVConfig(outer_splits=3, outer_repeats=1, inner_splits=3,
                           checkpoint_dir=cfg.paths.checkpoints),
        task="low_vs_high", pv=1, method=f"abl_{arm}")
    rows.append(r.assign(arm=arm))
abl = pd.concat(rows, ignore_index=True)
abl.to_parquet(cfg.paths.results / "ablations.parquet", index=False)
display(abl.groupby("arm")[["auc", "n_selected"]].mean())